# Multi-Category Benchmark on MVTec AD

This notebook evaluates the anomaly detection pipeline across multiple MVTec AD categories.

## Goals
- run the same pipeline on several categories
- compare AUROC values across categories
- identify easy and difficult object classes
- save benchmark results for analysis and reporting

## Imports

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from datasets.mvtec import MvtecAdDataset
from evaluation.metrics import compute_patchcore_scores
from models.patchcore import (
    FeatureExtractor,
    MultiScaleFeatureExtractor,
    build_memory_bank,
    random_coreset_sampling,
)
from src.config import DATA_DIR, BATCH_SIZE
from utils.normalization import preprocess_image

## Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Experiment Configuration

In [ ]:
CATEGORIES = [
    "capsule",
    "bottle",
    "screw",
    # "hazelnut",
    # "metal_nut",
    # "pill",
]

EXTRACTOR_TYPE = "multiscale"  # "layer3" or "multiscale"
SCORING_METHODS = ["mean", "max", "topk_mean"]
K_RATIO = 0.01

USE_RANDOM_CORESET = False
CORESET_RATIO = 0.1

SAVE_MEMORY_BANKS = False
SAVE_RESULTS = True

RUN_NAME = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_multicategory_patchcore"
print("Run:", RUN_NAME)

## Output folders

In [ ]:
OUTPUT_DIR = Path("outputs")
METRICS_DIR = OUTPUT_DIR / "metrics"
MEMORY_DIR = OUTPUT_DIR / "memory_banks"
TABLES_DIR = OUTPUT_DIR / "tables"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Extractor factory

In [ ]:
def build_extractor(extractor_type: str, device: torch.device):
    if extractor_type == "layer3":
        extractor = FeatureExtractor().to(device)
    elif extractor_type == "multiscale":
        extractor = MultiScaleFeatureExtractor().to(device)
    else:
        raise ValueError("extractor_type must be 'layer3' or 'multiscale'")

    extractor.eval()
    return extractor

## Single category evaluation function

In [ ]:
def evaluate_category(
    category: str,
    extractor_type: str,
    device: torch.device,
    use_random_coreset: bool = False,
    coreset_ratio: float = 0.1,
    k_ratio: float = 0.01,
):
    print(f"\n=== Category: {category} ===")

    train_dataset = MvtecAdDataset(
        root_dir=DATA_DIR,
        category=category,
        split="train",
        transform=preprocess_image,
    )

    test_dataset = MvtecAdDataset(
        root_dir=DATA_DIR,
        category=category,
        split="test",
        transform=preprocess_image,
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("Train size:", len(train_dataset))
    print("Test size:", len(test_dataset))

    extractor = build_extractor(extractor_type, device)

    memory_bank = build_memory_bank(
        feature_extractor=extractor,
        dataloader=train_loader,
        device=device,
    )

    full_memory_shape = tuple(memory_bank.shape)
    print("Full memory bank:", full_memory_shape)

    if use_random_coreset:
        memory_bank = random_coreset_sampling(
            memory_bank=memory_bank,
            sampling_ratio=coreset_ratio,
        )
        print("Coreset memory bank:", tuple(memory_bank.shape))

    category_results = {
        "category": category,
        "extractor_type": extractor_type,
        "train_size": len(train_dataset),
        "test_size": len(test_dataset),
        "memory_bank_shape": tuple(memory_bank.shape),
    }

    for method in SCORING_METHODS:
        scores, labels = compute_patchcore_scores(
            feature_extractor=extractor,
            dataloader=test_loader,
            memory_bank=memory_bank,
            device=device,
            reduction=method,
            k_ratio=k_ratio,
        )
        auc = roc_auc_score(labels, scores)
        category_results[method] = auc
        print(f"{method}: {auc:.4f}")

    return category_results, memory_bank

## Benchmark loop

In [ ]:
all_results = []

for category in CATEGORIES:
    category_results, memory_bank = evaluate_category(
        category=category,
        extractor_type=EXTRACTOR_TYPE,
        device=DEVICE,
        use_random_coreset=USE_RANDOM_CORESET,
        coreset_ratio=CORESET_RATIO,
        k_ratio=K_RATIO,
    )

    all_results.append(category_results)

    if SAVE_MEMORY_BANKS:
        save_path = MEMORY_DIR / f"{RUN_NAME}_{category}_{EXTRACTOR_TYPE}.pt"
        torch.save(
            {
                "category": category,
                "extractor_type": EXTRACTOR_TYPE,
                "memory_bank": memory_bank.cpu(),
            },
            save_path,
        )
        print("Saved memory bank to:", save_path)

## Results table 

In [ ]:
results_df = pd.DataFrame(all_results)
results_df

## Sort by best method

In [ ]:
results_df = results_df.sort_values(by="topk_mean", ascending=False)
results_df

## Smmary statistics

In [ ]:
summary = {
    "mean_of_mean": results_df["mean"].mean(),
    "mean_of_max": results_df["max"].mean(),
    "mean_of_topk_mean": results_df["topk_mean"].mean(),
}

summary

## Save result to csv/json file 

In [ ]:
if SAVE_RESULTS:
    csv_path = TABLES_DIR / f"{RUN_NAME}.csv"
    json_path = METRICS_DIR / f"{RUN_NAME}.json"

    results_df.to_csv(csv_path, index=False)

    payload = {
        "extractor_type": EXTRACTOR_TYPE,
        "use_random_coreset": USE_RANDOM_CORESET,
        "coreset_ratio": CORESET_RATIO if USE_RANDOM_CORESET else None,
        "k_ratio": K_RATIO,
        "categories": CATEGORIES,
        "summary": summary,
        "results": all_results,
    }

    with open(json_path, "w") as f:
        json.dump(payload, f, indent=4)

    print("Saved CSV to:", csv_path)
    print("Saved JSON to:", json_path)

## Best/ hardest categorues

In [ ]:
best_category = results_df.iloc[0]
worst_category = results_df.iloc[-1]

print("Best category (topk_mean):")
print(best_category[["category", "topk_mean"]])

print("\nWorst category (topk_mean):")
print(worst_category[["category", "topk_mean"]])